<a href="https://colab.research.google.com/github/ingkapat/t/blob/main/gendata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai tqdm -q

In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# ตั้งค่าจำนวน records ต่อ subtype รวม ~300
CONFIG = {
    'per_scam_short':     12,
    'per_scam_long':      10,
    'per_general_chat':    7,
    'per_verification':    8,
    'per_official':        8,
    'per_hard_negative':   9,
    'per_unknown':         5,

    'min_turns': 1,
    'max_turns': 6,

    'model':       'gpt-4o-mini',
    'temperature': 1.0,
    'max_retries': 3,
    'request_delay_sec': 0.15,

    'output_dir':  '/content/dataset_output',
    'output_file': 'dataset.jsonl',
}

# Subtypes แยกตามหมวด
SCAM_SHORT_SUBTYPES = [
    'reward', 'phishing', 'otp_hijack', 'sextortion', 'loan_scam', 'sms_alert',
]
SCAM_LONG_SUBTYPES = [
    'callcenter', 'investment', 'romance', 'tech_support',
    'job_scam', 'money_mule', 'impersonation',
]
GENERAL_CHAT_SUBTYPES   = ['friend_hangout', 'family_check', 'couple_talk', 'colleague']
VERIFICATION_SUBTYPES   = ['otp_thai', 'otp_english', 'reset_code']
OFFICIAL_SUBTYPES       = ['bank_real', 'gov_notice', 'hospital', 'company_hr']

# Hard negatives - เพิ่ม friend_new_number, colleague_expense, family_help
HARD_NEGATIVE_SUBTYPES  = [
    'real_otp', 'real_job', 'friend_borrow', 'real_parcel',
    'friend_new_number', 'colleague_expense', 'family_help',
]

UNKNOWN_SUBTYPES = ['silent_call', 'wrong_number', 'connection_issue', 'incomplete_speech']

# คำนวณจำนวน records ที่จะ generate
n_scam = (CONFIG['per_scam_short'] * len(SCAM_SHORT_SUBTYPES) +
          CONFIG['per_scam_long']  * len(SCAM_LONG_SUBTYPES))
n_chat = CONFIG['per_general_chat'] * len(GENERAL_CHAT_SUBTYPES)
n_ver  = CONFIG['per_verification'] * len(VERIFICATION_SUBTYPES)
n_off  = CONFIG['per_official']     * len(OFFICIAL_SUBTYPES)
n_hard = CONFIG['per_hard_negative']* len(HARD_NEGATIVE_SUBTYPES)
n_unk  = CONFIG['per_unknown']      * len(UNKNOWN_SUBTYPES)
n_total = n_scam + n_chat + n_ver + n_off + n_hard + n_unk

print('จำนวน records ที่จะสร้าง')
print(f'  scam (label=1)         : {n_scam}')
print(f'  general chat (label=0) : {n_chat}')
print(f'  verification (label=0) : {n_ver}')
print(f'  official (label=0)     : {n_off}')
print(f'  hard negative (label=0): {n_hard}')
print(f'  unknown (label=0)      : {n_unk}')
print(f'  รวม                    : {n_total}')
print(f'  สัดส่วน scam           : {n_scam/n_total*100:.1f}%')

In [ ]:
# Patterns สำหรับสร้างเนื้อหา

SCAM_SHORT_PATTERNS = {
    'reward':     'Notification claiming user won iPhone/cash/gold prize. '
                  'Requests contact via Line, link click, or ID card photo.',
    'phishing':   'SMS/notification about account locked or parcel delivery failure. '
                  'Pushes user to click bit.ly link or verify credentials.',
    'otp_hijack': 'Caller impersonates bank, requests 6-digit OTP code for verification. '
                  'Will use OTP to drain account.',
    'sextortion': 'Threatens to release private photos/videos unless payment is sent.',
    'loan_scam':  'Offers loan approval without credit check. '
                  'Requires upfront fee or ID document submission.',
    'sms_alert':  'Claims tax refund or social security benefit. '
                  'Requires dialing USSD code or submitting personal data.',
}

SCAM_LONG_PATTERNS = {
    'callcenter':    'Call center scam impersonating police, DSI, or revenue department. '
                     'Claims account is linked to criminal case. Demands transfer to safe account. '
                     'Forbids hanging up or telling anyone.',
    'investment':   'Investment scam guaranteeing 30-50% monthly return with no risk. '
                     'Requests user to chat or click link to register and transfer funds urgently.',
    'romance':       'Romance scam with affectionate messages. '
                     'Claims to be foreign businessman, soldier, or doctor. '
                     'Eventually requests emergency loan transfer.',
    'tech_support':  'Impersonates Microsoft, Apple, or bank technical support. '
                     'Reports virus or hacked account. '
                     'Instructs user to install AnyDesk or TeamViewer for remote access.',
    'job_scam':      'Online job offer for liking videos or completing tasks. '
                     'Initially pays small amounts, then requires deposits to unlock withdrawals.',
    'money_mule':    'Offers commission for receiving and forwarding bank transfers. '
                     'Claims to be foreign company needing local Thai accounts.',
    'impersonation': 'Impersonates friend or relative claiming new phone number. '
                     'Creates emergency requiring urgent money transfer. '
                     'Refuses callbacks to old number. No personal context, vague identity.',
}

GENERAL_CHAT_PATTERNS = {
    'friend_hangout': 'Friend invites for shopping, food, or movies. Casual language with กู มึง แก เธอ.',
    'family_check':   'Family member checking on user, asking about meals or work. Warm tone.',
    'couple_talk':    'Partner calling to chat, set meeting, or express affection.',
    'colleague':      'Coworker discussing meetings, work tasks, or lunch plans.',
}

VERIFICATION_PATTERNS = {
    'otp_thai':    'Automated SMS/voice message in Thai delivering OTP code. '
                   'Includes warning not to share with anyone.',
    'otp_english': 'Automated message in English. Format: '
                   '"Your verification code is XXXXXX. Do not share."',
    'reset_code':  'Password reset confirmation code or 2FA code delivery.',
}

OFFICIAL_PATTERNS = {
    'bank_real':  'Bank notification about credit card statement, expiring card, or transaction confirmation. '
                  'Does not request OTP or sensitive information.',
    'gov_notice': 'Government office (revenue dept, social security, transport) announcing rights or appointments. '
                  'Does not request payment.',
    'hospital':   'Hospital calling for appointment scheduling, lab results, or rescheduling.',
    'company_hr': 'Real company HR scheduling job interview. Identifies company name and position clearly.',
}

# Hard negatives - กลุ่มสำคัญสำหรับโมเดลให้แยก scam vs ไม่ใช่
HARD_NEGATIVE_PATTERNS = {
    'real_otp':      'Real bank requesting OTP for transaction confirmation. '
                     'Includes reference number. Does not ask for additional credentials.',

    'real_job':      'Real HR scheduling urgent interview because position closing soon. '
                     'Does not request payment or ID documents.',

    'friend_borrow': 'Close friend genuinely borrowing money. Uses informal pronouns กู/มึง. '
                     'Discusses other topics first. Provides personal context. '
                     'Does not pressure if user wants to verify.',

    'real_parcel':   'Real delivery service notifying about package. Includes tracking number. '
                     'Does not send links or request payment.',

    # subtype ใหม่ - เพื่อนจริงเปลี่ยนเบอร์ ขอเงิน (ตรงข้ามกับ impersonation scam)
    'friend_new_number':
                     'Real friend with new phone number asking to borrow small amount. '
                     'Provides clear personal context (school name, shared memories, '
                     'mutual friends, past events). Reason for new number is mundane '
                     '(lost phone, broken phone, switched carrier). Not pressuring. '
                     'Willing to wait for user to verify through old number or mutual friend. '
                     'Small amount (200-1000 baht). No urgency about transfer timing.',

    'colleague_expense':
                     'Coworker asking to split bill or borrow small amount for lunch/coffee/transport. '
                     'References specific workplace context (project name, team members, office events). '
                     'Casual tone. Small amount. Easy to verify identity.',

    'family_help':
                     'Family member (parent/sibling/relative) asking for financial help with '
                     'real life situation. References family events or shared knowledge. '
                     'Conversational pace, not pressuring. Mentions specific reason '
                     '(utility bill, school fees, medical appointment).',
}

UNKNOWN_PATTERNS = {
    'silent_call':       'Caller does not speak. May only have background noise or breathing. '
                          'User says ฮัลโหล or asks who is calling. No response.',
    'wrong_number':      'Caller asks for someone who is not at this number. '
                          'Brief exchange clarifying wrong number. Caller hangs up or apologizes.',
    'connection_issue':  'Phone connection has noise, echo, or breaking up. '
                          'Conversation cannot proceed. Both parties say cannot hear.',
    'incomplete_speech': 'Caller says incomplete sentences, mumbles, or speech is cut off. '
                          'Content is unclear and cannot be classified.',
}

# Variation matrix: บุคลิกเหยื่อและสไตล์ภาษา
VICTIM_STYLES = {
    'naive':     'Trusting and easily persuaded. Brief positive responses like อือ, อ๋อ, โอเค, จริงเหรอ.',
    'skeptical': 'Mildly suspicious. Asks clarifying questions like แบบว่า, เอ๊ะ, รอก่อนนะ, แน่ใจเหรอ.',
    'aware':     'Recognizes the scam pattern. Cuts off with เฮ้ย หยุดก่อน, ไม่เอาแล้ว, จะวางสายนะ.',
    'busy':      'Busy or in a hurry. Short responses like รีบหน่อย, แป๊บนึง, กำลังขับรถ, มีอะไรเร็วๆ.',
    'elderly':   'Older person, slow to respond. Says อะไรนะ, พูดอีกทีได้ไหม, ฟังไม่ค่อยชัด.',
}

LANGUAGE_STYLES = {
    'formal': 'Standard Thai with polite particles ครับ/ค่ะ.',
    'casual': 'Colloquial Thai with particles อ่ะ, นะ, จ้า, เอ้า, อือ. Short sentences.',
    'mixed':  'Mixed Thai-English code-switching. Words like OK, confirm, check, sure.',
}

print('โหลด patterns สำเร็จ')

In [ ]:
# System prompt และ prompt builder

SYSTEM_PROMPT = '''You are a Thai language conversation writer specializing in realistic phone dialogues.
Your task is to generate synthetic phone conversations for training a scam detection AI.

Critical requirement: Generated dialogue must sound like authentic Thai phone conversations,
not formal written language.

Guidelines:

1. Use realistic spoken Thai with appropriate sentence-final particles:
   ครับ, ค่ะ, นะ, น่ะ, อ่ะ, เว้ย, จ้า, ดิ
   อือ, อ๋อ, อา, แหม, เฮ้ย, อ้าว, โอ้โห
   แบบว่า, คือ, งั้น, เดี๋ยว

2. For close friends and family relationships, use informal pronouns:
   กู, มึง, เอ็ง, แก, เธอ, ไอ้..., อี่...

3. Keep utterances short and natural, not full sentences:
   Bad:  "ผมคิดว่าฟังดูน่าสนใจมากเลยครับ"
   Good: "อือ น่าสนใจอะ"

4. Scammers typically use overly formal language with words like กรุณา, โปรด, เรียน.

5. The user (call recipient) typically speaks less than the caller, especially in scam scenarios.

6. Some conversations may have only the caller speaking (broadcast/SMS style),
   with no user response. This is acceptable.

7. For real friend with new number scenarios: caller must provide concrete personal context
   (specific names, places, shared events). Caller does not pressure if user wants to verify.

Reference examples:

Friend invitation:
  [caller: มึงว่างป่ะ ไปกินชาบูกัน]
  [user: ได้ๆ กี่โมงล่ะ]
  [caller: สัก 6 โมงเย็น]

OTP delivery (single-turn):
  [caller: รหัส OTP ของคุณคือ 123456 กรุณาอย่าเผยแพร่]

Scam impersonation (vague identity, urgent):
  [caller: เฮ้ย กูเปลี่ยนเบอร์ใหม่ รีบโอนเงินให้กูหน่อย ติดเรื่องด่วน]
  [user: ใครวะ]
  [caller: กูเองไง รีบโอนมาก่อน]

Real friend with new number (specific context, no pressure):
  [caller: เฮ้ย กูบอลเอง เปลี่ยนเบอร์ใหม่อ่ะ มือถือเครื่องเก่ามันเสีย]
  [user: อ่อ ไอ้บอล จำได้ ตอน ม.ปลายเล่นบอลด้วยกัน]
  [caller: ใช่ๆ ขอยืม 500 หน่อยได้ป่ะ ลืมเอาตังมา ไว้กินข้าวเที่ยง]
  [user: ได้ๆ เดี๋ยวโอนให้]

Output format: JSON object with key "turns" only. No additional text.'''

TURNS_HINT = '{"turns": [{"speaker": "caller", "text": "..."}]}'

def build_prompt(category, subtype, victim_style, lang_style, min_t, max_t, extra=''):
    victim_section = f'Recipient profile: {victim_style}\n' if victim_style else ''

    return f'''Generate one phone conversation:

Category: {category}
Subtype: {subtype}
{victim_section}Language style: {lang_style}
Target length: {min_t}-{max_t} turns
{extra}

Language must sound natural for Thai phone conversation, not written prose.

Output JSON: {TURNS_HINT}'''

print('พร้อม build prompt')

In [ ]:
# API client และ helpers
from openai import OpenAI
import json, time

client = OpenAI()

def call_gpt(prompt, max_tokens=1000):
    response = client.chat.completions.create(
        model=CONFIG['model'],
        temperature=CONFIG['temperature'],
        max_tokens=max_tokens,
        response_format={'type': 'json_object'},
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt},
        ],
    )
    return response.choices[0].message.content.strip()

def parse_turns(text):
    data = json.loads(text)
    turns = data['turns'] if isinstance(data, dict) and 'turns' in data else data
    assert isinstance(turns, list) and len(turns) >= 1
    for t in turns:
        assert t['speaker'] in ('caller', 'user')
        assert isinstance(t['text'], str) and t['text'].strip()
    return turns

def generate_one(task):
    prompt = build_prompt(
        category=task['category'],
        subtype=task['desc'],
        victim_style=task.get('victim_desc'),
        lang_style=task['lang_desc'],
        min_t=task['min_t'],
        max_t=task['max_t'],
        extra=task.get('extra', ''),
    )

    for attempt in range(CONFIG['max_retries']):
        try:
            raw   = call_gpt(prompt)
            turns = parse_turns(raw)
            return {'label': task['label'], 'turns': turns}
        except Exception as e:
            print(f'  attempt {attempt+1} ล้มเหลว: {e}')
            time.sleep(2 ** attempt)
    return None

# ทดสอบการเชื่อมต่อ API
try:
    test_response = call_gpt('Return JSON: {"status": "ok"}', max_tokens=50)
    print(f'เชื่อมต่อ API สำเร็จ')
    print(f'response: {test_response[:80]}')
except Exception as e:
    print(f'เชื่อมต่อ API ล้มเหลว: {e}')

In [ ]:
# สร้าง task list ด้วย variation matrix
import itertools, random

random.seed(42)

victim_keys = list(VICTIM_STYLES.keys())
lang_keys   = list(LANGUAGE_STYLES.keys())
matrix      = list(itertools.product(victim_keys, lang_keys))

tasks = []

def add_scam_short_tasks():
    for st in SCAM_SHORT_SUBTYPES:
        for i in range(CONFIG['per_scam_short']):
            v_key, l_key = matrix[i % len(matrix)]
            tasks.append({
                'label': 1,
                'category':    f'Scam (short SMS-style) - {st}',
                'desc':        SCAM_SHORT_PATTERNS[st],
                'victim_desc': VICTIM_STYLES[v_key],
                'lang_desc':   LANGUAGE_STYLES[l_key],
                'min_t': 1, 'max_t': 3,
            })

def add_scam_long_tasks():
    for st in SCAM_LONG_SUBTYPES:
        for i in range(CONFIG['per_scam_long']):
            v_key, l_key = matrix[i % len(matrix)]
            tasks.append({
                'label': 1,
                'category':    f'Scam (multi-turn) - {st}',
                'desc':        SCAM_LONG_PATTERNS[st],
                'victim_desc': VICTIM_STYLES[v_key],
                'lang_desc':   LANGUAGE_STYLES[l_key],
                'min_t': 3, 'max_t': 6,
            })

def add_general_chat_tasks():
    for st in GENERAL_CHAT_SUBTYPES:
        for i in range(CONFIG['per_general_chat']):
            _, l_key = matrix[i % len(matrix)]
            tasks.append({
                'label': 0,
                'category':  'General chat',
                'desc':      GENERAL_CHAT_PATTERNS[st],
                'lang_desc': LANGUAGE_STYLES[l_key],
                'min_t': 2, 'max_t': 5,
                'extra': 'Use informal pronouns where appropriate.',
            })

def add_verification_tasks():
    for st in VERIFICATION_SUBTYPES:
        for _ in range(CONFIG['per_verification']):
            tasks.append({
                'label': 0,
                'category':  'Verification code',
                'desc':      VERIFICATION_PATTERNS[st],
                'lang_desc': 'Automated formal style',
                'min_t': 1, 'max_t': 1,
                'extra': 'Caller-only message. No user response.',
            })

def add_official_tasks():
    for st in OFFICIAL_SUBTYPES:
        for i in range(CONFIG['per_official']):
            _, l_key = matrix[i % len(matrix)]
            tasks.append({
                'label': 0,
                'category':  'Official call',
                'desc':      OFFICIAL_PATTERNS[st],
                'lang_desc': LANGUAGE_STYLES[l_key],
                'min_t': 1, 'max_t': 4,
                'extra': 'Must not request OTP, payment, or sensitive data.',
            })

def add_hard_negative_tasks():
    for st in HARD_NEGATIVE_SUBTYPES:
        for i in range(CONFIG['per_hard_negative']):
            _, l_key = matrix[i % len(matrix)]

            # extra instructions เฉพาะ subtype
            if st == 'friend_borrow':
                extra = 'Use informal pronouns. Discuss unrelated topics first.'
            elif st == 'friend_new_number':
                extra = ('Caller must include specific personal context like school name, '
                         'mutual friend names, or shared past events. '
                         'Reason for new number must be mundane (lost/broken phone, switched carrier). '
                         'Caller must NOT pressure user. Amount must be small (200-1000 baht). '
                         'If user asks to verify, caller agrees willingly.')
            elif st == 'colleague_expense':
                extra = ('Reference specific workplace context such as team name, project, '
                         'office event, or coworker names. Small amount only.')
            elif st == 'family_help':
                extra = ('Reference family-specific context such as relative names, family events, '
                         'or household situations. Conversational pace, no pressure.')
            else:
                extra = ''

            tasks.append({
                'label': 0,
                'category':  f'Hard negative - {st}',
                'desc':      HARD_NEGATIVE_PATTERNS[st],
                'lang_desc': LANGUAGE_STYLES[l_key],
                'min_t': 2, 'max_t': 5,
                'extra': extra,
            })

def add_unknown_tasks():
    for st in UNKNOWN_SUBTYPES:
        for _ in range(CONFIG['per_unknown']):
            tasks.append({
                'label': 0,
                'category':  f'Unknown - {st}',
                'desc':      UNKNOWN_PATTERNS[st],
                'lang_desc': 'Variable - depends on situation',
                'min_t': 1, 'max_t': 3,
                'extra': 'Content should be brief, unclear, or interrupted. '
                         'Not a real conversation.',
            })

add_scam_short_tasks()
add_scam_long_tasks()
add_general_chat_tasks()
add_verification_tasks()
add_official_tasks()
add_hard_negative_tasks()
add_unknown_tasks()

random.shuffle(tasks)

from collections import Counter
label_dist = Counter(t['label'] for t in tasks)
print(f'จำนวน task ทั้งหมด: {len(tasks)}')
print(f'  label=1 (scam)    : {label_dist[1]} ({label_dist[1]/len(tasks)*100:.1f}%)')
print(f'  label=0 (not-scam): {label_dist[0]} ({label_dist[0]/len(tasks)*100:.1f}%)')

In [ ]:
# generate dataset
from tqdm.notebook import tqdm

os.makedirs(CONFIG['output_dir'], exist_ok=True)
DATASET_PATH = os.path.join(CONFIG['output_dir'], CONFIG['output_file'])

successful = 0
failed = 0

with open(DATASET_PATH, 'w', encoding='utf-8') as f:
    for task in tqdm(tasks, desc='Generating'):
        record = generate_one(task)
        if record is None:
            failed += 1
            continue
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
        f.flush()
        successful += 1
        time.sleep(CONFIG['request_delay_sec'])

print()
print(f'สร้างเสร็จ')
print(f'  สำเร็จ : {successful}')
print(f'  ล้มเหลว: {failed}')
print(f'  ไฟล์   : {DATASET_PATH}')

In [ ]:
# สถิติของ dataset
records = []
with open(DATASET_PATH, encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))

n_total = len(records)
n_scam  = sum(1 for r in records if r['label'] == 1)
n_legit = n_total - n_scam

turn_counts  = [len(r['turns']) for r in records]
text_lengths = [sum(len(t['text']) for t in r['turns']) for r in records]

len_scam  = [text_lengths[i] for i, r in enumerate(records) if r['label'] == 1]
len_legit = [text_lengths[i] for i, r in enumerate(records) if r['label'] == 0]

print('สถิติ dataset')
print(f'  จำนวนทั้งหมด        : {n_total}')
print(f'  สัดส่วน label       : scam={n_scam} ({n_scam/n_total*100:.1f}%), '
      f'not-scam={n_legit} ({n_legit/n_total*100:.1f}%)')
print(f'  จำนวน turn          : min={min(turn_counts)}, max={max(turn_counts)}, '
      f'mean={sum(turn_counts)/n_total:.2f}')
print(f'  ความยาว text (ตัวอักษร): min={min(text_lengths)}, max={max(text_lengths)}, '
      f'mean={sum(text_lengths)/n_total:.1f}')
print(f'  ความยาวเฉลี่ย scam    : {sum(len_scam)/len(len_scam):.1f}')
print(f'  ความยาวเฉลี่ย not-scam: {sum(len_legit)/len(len_legit):.1f}')

length_diff = abs(sum(len_scam)/len(len_scam) - sum(len_legit)/len(len_legit))
length_ratio = length_diff / (sum(len_legit)/len(len_legit)) * 100
if length_ratio > 30:
    print(f'  เตือน: ความยาวต่างกัน {length_ratio:.0f}% อาจเกิด length leak')

In [ ]:
# ดูตัวอย่างแบบสุ่ม
import random as rd
rd.seed(0)

preview_indices = rd.sample(range(len(records)), 5)

for idx in preview_indices:
    r = records[idx]
    label_text = 'SCAM' if r['label'] == 1 else 'NOT_SCAM'
    print('-' * 60)
    print(f'index: {idx} | label: {r["label"]} ({label_text}) | turns: {len(r["turns"])}')
    for t in r['turns']:
        print(f'  [{t["speaker"]:6s}] {t["text"]}')
    print()

In [ ]:
# โหลดไฟล์
from google.colab import files
files.download(DATASET_PATH)